In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# Read the raw files from data/raw/
subscriptions = pd.read_csv('../data/raw/subscriptions.csv')
usage = pd.read_csv('../data/raw/usage.csv')

# Merge the two dataframes on customer_id
merged = pd.merge(subscriptions, usage, on='customer_id')

# Save the merged dataset into data/
merged.to_csv('../data/raw/merged.csv', index=False)

In [ ]:
df = pd.read_csv('../data/raw/merged.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.drop_duplicates(inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

In [ ]:
df.drop(columns=['churn_date','cancel_reason','customer_id'], inplace=True)
df = df.dropna(subset=['minutes_active','churned'])

In [ ]:
# Display unique value counts for each column
for column in df.columns:
    
    unique_count = len(df[column].unique())
    print(f"\n{column} has {unique_count} unique values.")

    if unique_count <= 20:
        print(f"Unique values in {column}: {df[column].unique()}")

In [ ]:
df.drop_duplicates(inplace=True)


In [ ]:
df.duplicated().sum()

In [ ]:
df['plan_tier'] = df['plan_tier'].replace({'Basic':'basic', 'Premium':'premium','Plus':'plus'})


In [ ]:
df=df.dropna(subset=["workouts_completed"])
df=df.fillna(df.mode().iloc[0])

# Define custom bins
bins = [0, 10, 20, 30, 40,48]
labels = ['0-9', '10-19', '20-29', '30-40','40-48']

# Apply binning
df['workouts_completed'] = pd.cut(df['workouts_completed'], bins=bins, labels=labels, right=False)
workout_mapping = {'0-9': 0, '10-19': 1, '20-29': 2, '30-40': 3, '40-48': 4}
df['workouts_completed'] = df['workouts_completed'].map(workout_mapping)

print(df)

In [ ]:
from sklearn.preprocessing import LabelEncoder
# List of columns to encode
columns_to_encode = ['churned', 'plan_type','plan_tier','acquisition_channel']
pd.to_numeric(columns_to_encode,errors='coerce')

# Create a dictionary to hold LabelEncoder instances for each column
label_encoders = {column: LabelEncoder() for column in columns_to_encode}

# Apply LabelEncoder to each column
for column in columns_to_encode:
    df[column] = label_encoders[column].fit_transform(df[column])

# Check the first few rows of the DataFrame
print(df.head())

In [ ]:

from sklearn.preprocessing import OrdinalEncoder
ordinal_encoder = OrdinalEncoder()

# Fit and transform the 'age_group' column
df['age_group'] = ordinal_encoder.fit_transform(df[['age_group']])
df['monthly_price'] = ordinal_encoder.fit_transform(df[['monthly_price']])


print(df.sample(5))

In [ ]:
df

In [ ]:

df['subscription_month'] = pd.to_datetime(df['month'], format='%Y-%m', errors='coerce')
df['subscription_year'] = df['subscription_month'].dt.year
df['subscription_month'] = df['subscription_month'].dt.month
df.drop(columns='month',inplace=True)


In [ ]:

df['signup_date'] = pd.to_datetime(df['signup_date'], format='%Y-%m-%d', errors='coerce')
df['signup_year'] = df['signup_date'].dt.year
df['signup_month'] = df['signup_date'].dt.month
df.drop(columns='signup_date',inplace=True)

In [ ]:

# Group the DataFrame by 'primary_device'
grouped_df = df.groupby('primary_device')

# Calculate the churn rate for each group
churn_rate = grouped_df['churned'].mean()

# Reset the index to make 'primary_device' a column again
churn_rate = churn_rate.reset_index()

# Rename the columns for clarity
churn_rate.rename(columns={'churned': 'churn_rate'}, inplace=True)

# Convert churn rate to percentage
churn_rate['churn_rate'] = churn_rate['churn_rate'] * 100

# Check the first few rows of the result
print(churn_rate.head())

# Churn Rate Analysis by Primary Device

The churn rate across different primary devices reveals the following insights:

- **Android**:
  - **Churn Rate**: **37.99%**
  - **Analysis**: Customers using Android devices have the highest churn rate, indicating potential issues with Android-specific features or user experience.

- **iOS**:
  - **Churn Rate**: **33.16%**
  - **Analysis**: Customers using iOS devices have a moderate churn rate, suggesting a relatively balanced experience with both iOS and Android.

- **Web**:
  - **Churn Rate**: **36.58%**
  - **Analysis**: Customers using web-based services have a churn rate that is slightly higher than iOS but lower than Android, possibly due to differences in user interaction and support.

These differences in churn rates suggest that there might be specific issues or preferences associated with each device type that contribute to higher churn among certain groups. Further analysis may be required to identify the underlying reasons for these churn rates and implement strategies to mitigate them.

---

This markdown format uses bullet points and annotations to clearly highlight the churn rates and provide additional analysis and insights for each device type.

In [ ]:

# Convert the 'primary_Device' column to category type
df['primary_device'] = df['primary_device'].astype('category')
# Apply label encoding to the 'primary_Device' column
df['primary_device'] = df['primary_device'].cat.codes
# Display the updated dataframe
df.primary_device.sample(5)

In [ ]:
df.sample(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



plt.figure(figsize=(16, 16))
correlation_matrix = df.select_dtypes(include='number').corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.show()



`plan_tier` and `monthly_price_category` columns are nearly correlated that means we can drop any of two and use one instead.i would choose `monthly_price_category` to work with. 

In [ ]:
df.drop(columns= 'plan_tier',inplace=True)

In [ ]:
df.to_csv('../data/raw/cleaned_data.csv',index=False)